# Configs — 04: Self-Discovery, Slim Mode, Field-by-Field Docs

**Persona:** Catalog/Collection admin

**Goal:** Demonstrate the standards-aligned self-discovery features added to the composed-config API:

1. **JSON Hyper-Schema `_links`** at the response root — discover supported query parameters and alternate representations without consulting OpenAPI.
2. **`?docs=field`** (default) — lightweight `{field_name: description}` map per class.
3. **`?docs=schema`** — full JSON Schema 2020-12 with `examples` per field (for form-builders).
4. **`?include=scope`** (default) — body shows only configs owned by this scope; upstream-tier configs go to `inherited`.
5. **`?include=upstream`** — verbose mode (full waterfall in body).
6. **Per-class `entity` annotation** distinguishing items / collection / assets driver configs.

The endpoints are unchanged from earlier notebooks; the additions surface as new fields in the response and one new query parameter.

In [ ]:
import os
import json
import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL      = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
ADMIN_TOKEN   = os.environ.get("DYNASTORE_ADMIN_TOKEN", "")
CATALOG_ID    = os.environ.get("CATALOG_ID", "demo-catalog")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "sentinel2-l2a")

headers = {"Authorization": f"Bearer {ADMIN_TOKEN}"} if ADMIN_TOKEN else {}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=120)

print(f"Base URL    : {BASE_URL}")
print(f"Catalog     : {CATALOG_ID}")
print(f"Collection  : {COLLECTION_ID}")

## Step 1 — Slim default response (`?include=scope`, the new default)

Asking the collection scope returns ONLY configs owned by this collection — `_visibility=collection` configs (routing + driver-tier with sidecars) plus any explicit collection-level overrides. Upstream-tier configs are summarised in `inherited` (class_key → source) — operators see what exists without 700 rows of code defaults.

In [ ]:
resp = client.get(f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}")
assert resp.status_code == 200, resp.text
body = resp.json()

print("Top-level keys:", sorted(body.keys()))
print(f"\nClasses in body  : {sum(len(v) if isinstance(v, dict) else 0 for v in body['configs'].values())}")
print(f"Inherited count  : {len(body.get('inherited') or {})}")
print(f"\nSample inherited (class_key → source):")
for k, src in list((body.get('inherited') or {}).items())[:5]:
    print(f"  {k:40s} ← {src}")

## Step 2 — Self-discovery via `_links` (JSON Hyper-Schema)

Every response carries a `_links` block. The `self` link's `hrefSchema` is a JSON Schema 2020-12 doc describing every query parameter with `description` + `examples`. The `alternate` links advertise the other representations (full waterfall, schema mode, lean mode, delta-only). The `edit` link is a URI Template for per-class PATCH.

In [ ]:
for link in body["_links"]:
    print(f"{link['rel']:12s} {link['method']:6s} {link['href']}")
    if link.get('title'):
        print(f"             ↳ {link['title']}")

# The self link's hrefSchema lists every supported query param
self_link = next(l for l in body["_links"] if l["rel"] == "self")
print("\n=== Query parameters discoverable via hrefSchema ===")
for name, spec in self_link["hrefSchema"]["properties"].items():
    examples = spec.get("examples", [])
    print(f"  ?{name}={spec.get('default')!r:12} (examples: {examples})")
    print(f"     {spec['description']}\n")

## Step 3 — Field-by-field documentation (`?docs=field`, the default mode)

Each class in the response gets a `meta.{ClassName}.field_docs` map: `{field_name: description}`. Lightweight (no full JSON Schema) — useful for tooltips and inline help. Combine with `?meta=true` to also see source diagnostics.

In [ ]:
resp = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
    params={"meta": "true"},
)
body = resp.json()

# Show field-by-field docs for the items driver config
items_meta = body["meta"].get("items_postgresql_driver")
if items_meta:
    print("=== ItemsPostgresqlDriverConfig — field-level docs ===")
    print(f"source  : {items_meta['source']}")
    print(f"entity  : {items_meta['entity']}")
    for field_name, desc in (items_meta.get('field_docs') or {}).items():
        print(f"\n  {field_name}:\n    {desc[:140]}{'...' if len(desc) > 140 else ''}")

## Step 4 — Full JSON Schema with `examples` (`?docs=schema`, for form-builders)

Switching to `?docs=schema` swaps the lightweight `field_docs` map for the full JSON Schema 2020-12 document per class. Each `Field(...)` declaration's `description` AND `examples` come through — operators see realistic values to seed forms, and the schema can be fed directly to a JSON-Schema-aware form-builder.

In [ ]:
resp = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
    params={"docs": "schema"},
)
body = resp.json()

# Show write-policy schema with examples — this is where Change 6's worked
# scenarios (external-id versioning, geohash dedup, batch idempotency) live
wp = body["meta"].get("collection_write_policy")
if wp and wp.get("json_schema"):
    schema = wp["json_schema"]
    print("=== CollectionWritePolicy — schema with examples ===\n")
    for field_name, spec in schema["properties"].items():
        examples = spec.get("examples")
        if examples is not None:
            print(f"  {field_name:30s} examples = {examples}")

## Step 5 — Collection example: contextualizing sidecars by entity

PostgreSQL drivers carry sidecars (geometries, attributes, item-metadata, etc.). A single collection can have items-side and collection-metadata-side sidecars on different drivers. The `entity` annotation on each driver's meta entry tells you which entity bucket it governs.

In [ ]:
resp = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
    params={"meta": "true"},
)
body = resp.json()

# Walk driver-tier configs and group by entity
drivers = body["configs"].get("storage", {}).get("drivers", {})
print("=== Driver-tier configs grouped by entity (from `_address[2]`) ===\n")
for entity_bucket, classes in drivers.items():
    print(f"--- entity: {entity_bucket} ---")
    for class_key, payload in classes.items():
        sidecars = payload.get("sidecars", [])
        meta_entry = body["meta"].get(class_key, {})
        print(f"  {class_key} (entity={meta_entry.get('entity')!r})")
        if sidecars:
            for sc in sidecars:
                print(f"    + sidecar: {sc.get('sidecar_type', sc)} (enabled={sc.get('enabled', '?')})")
    print()

## Step 6 — Asset example: routing + driver config at catalog scope

Asset routing config lives at catalog scope. The same self-discovery patterns (`_links`, `?docs=field`, `?include=scope`) apply uniformly — the only thing that changes is which configs the slim filter keeps in body (catalog-vis at catalog scope) vs surfaces in `inherited` (platform-only).

In [ ]:
resp = client.get(
    f"/configs/catalogs/{CATALOG_ID}",
    params={"meta": "true"},
)
body = resp.json()

# Asset routing config at catalog scope
routing = body["configs"].get("storage", {}).get("routing", {})
asset_routing = routing.get("asset_routing_config")
if asset_routing:
    print("=== AssetRoutingConfig — operations dispatch ===")
    for op, entries in asset_routing.get("operations", {}).items():
        for e in entries:
            print(f"  {op:8s} → {e.get('driver_id'):30s} on_failure={e.get('on_failure')}")

# Edit link (templated) — operators substitute {class_key} to PATCH a single class
edit_link = next((l for l in body['_links'] if l['rel'] == 'edit'), None)
if edit_link:
    print(f"\nEdit URL template: {edit_link['href']}")
    print(f"  Example: PATCH {edit_link['href'].replace('{class_key}', 'asset_routing_config')}")

## Step 7 — Opt-in to verbose mode (`?include=upstream`)

Existing scripts/dashboards that consume the full waterfall can opt back into the pre-slim default with `?include=upstream`. Compare line counts to see the effect.

In [ ]:
slim = client.get(f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}").text
full = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
    params={"include": "upstream"},
).text

print(f"Slim mode (default)   : {len(slim):>7,d} bytes")
print(f"Upstream (verbose)    : {len(full):>7,d} bytes")
print(f"Reduction             : {(1 - len(slim)/len(full))*100:>6.1f}% smaller")

## Summary

- `_links` (JSON Hyper-Schema) at every response root — operators discover params + alternates without OpenAPI.
- `?docs=field` (default) → `meta.{class}.field_docs` map per class.
- `?docs=schema` → full JSON Schema 2020-12 with `examples`; ready for form-builders.
- `?include=scope` (default) → slim body, `inherited` summary for upstream tiers.
- `?include=upstream` → today's verbose mode for backward-compat.
- `meta.{class}.entity` distinguishes items / collection / assets driver-tier configs at a glance.
- Write-policy fields now carry `examples` per field — ranging from realistic external-id paths (`'properties.code'`) to geohash precisions (6/9/12) — surfaced through both `field` and `schema` docs modes.